# English-German Bidirectional PBSMT Demo Training

This notebook builds a strict phrase-based SMT demo system for:

1. English -> German
2. German -> English

It does not require manual dataset upload. It downloads the corpora, builds a small subset, trains both directional models, evaluates them, and packages the artifacts.

Assumptions:

1. Run in Google Colab on Linux.
2. CPU runtime is enough. GPU is not required.
3. This is a fast demo baseline, so it skips tuning to keep the workflow smaller and more reproducible.


In [ ]:
import json
from pathlib import Path

BASE = Path('/content/pbsmt_demo')
DATA = BASE / 'data'
RAW = DATA / 'raw'
WORK = DATA / 'work'
PROC = DATA / 'processed'
TOOLS = BASE / 'tools'
MODELS = BASE / 'models'
EXPORTS = BASE / 'exports'

for path in [BASE, DATA, RAW, WORK, PROC, TOOLS, MODELS, EXPORTS]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'news_pairs': 25000,
    'europarl_pairs': 15000,
    'total_pairs': 40000,
    'train_pairs': 36000,
    'valid_pairs': 2000,
    'test_pairs': 2000,
    'max_sentence_length': 50,
    'threads': 2,
    'lm_order': 5,
    'max_phrase_length': 5,
    'alignment_heuristic': 'grow-diag-final-and',
}

print(json.dumps(CONFIG, indent=2))


## 1. Install dependencies

This installs build tools plus the packages usually needed for compiling the classical SMT toolchain in Colab.


In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
grep -RIl '^deb-src .*r2u\.stat\.illinois\.edu' /etc/apt/sources.list /etc/apt/sources.list.d 2>/dev/null | while read -r file; do sed -i '/^deb-src .*r2u\.stat\.illinois\.edu/s/^/# /' "$file"; done || true
apt-get -qq update
apt-get -qq install -y build-essential cmake git wget curl xz-utils zlib1g-dev libbz2-dev liblzma-dev libboost-all-dev pkg-config automake libtool gawk unzip csh
python3 -m pip install -q sacrebleu


## 2. Clone and build the SMT toolchain

This notebook builds:

1. Moses
2. KenLM
3. fast_align
4. GIZA++ / mkcls

`fast_align` is used for alignment in this demo. `GIZA++` and `mkcls` are still built because parts of the classical Moses ecosystem expect those utilities to exist.


In [ ]:
%%bash
set -e
BASE=/content/pbsmt_demo
TOOLS=$BASE/tools
cd "$TOOLS"
[ -d mosesdecoder ] || git clone --depth 1 https://github.com/moses-smt/mosesdecoder.git
[ -d kenlm ] || git clone --depth 1 https://github.com/kpu/kenlm.git
[ -d fast_align ] || git clone --depth 1 https://github.com/clab/fast_align.git
[ -d giza-pp ] || git clone --depth 1 https://github.com/moses-smt/giza-pp.git
make -C "$TOOLS/giza-pp/GIZA++-v2" -j2
make -C "$TOOLS/giza-pp/mkcls-v2" -j2
mkdir -p "$TOOLS/kenlm/build"
cd "$TOOLS/kenlm/build"
cmake -Wno-dev .. >/dev/null
make -j2 >/dev/null
mkdir -p "$TOOLS/fast_align/build"
cd "$TOOLS/fast_align/build"
cmake -Wno-dev .. >/dev/null
make -j2 >/dev/null
cd "$TOOLS/mosesdecoder"
git describe --dirty >/dev/null 2>&1 || git tag -f colab-build >/dev/null 2>&1 || true
./bjam -j2 --without-tcmalloc --no-xmlrpc-c


In [ ]:
import os
import shutil
from pathlib import Path

tools = Path('/content/pbsmt_demo/tools')
moses_root = tools / 'mosesdecoder'
repo_bin = moses_root / 'bin'
repo_bin.mkdir(parents=True, exist_ok=True)
external_bin = tools / 'bin'
external_bin.mkdir(parents=True, exist_ok=True)

def find_exec(root: Path, name: str):
    candidates = [p for p in root.rglob(name) if p.is_file() and os.access(p, os.X_OK)]
    if not candidates:
        raise FileNotFoundError(f'Missing executable: {name}')
    candidates.sort(key=lambda p: (0 if '/bin/' in str(p) else 1, len(str(p))))
    return candidates[0]

required_moses_bins = ['moses', 'extract', 'score', 'lexical-reordering-score', 'symal']
resolved = {}
for binary_name in required_moses_bins:
    resolved[binary_name] = find_exec(moses_root, binary_name)
    destination = repo_bin / binary_name
    if resolved[binary_name].resolve() != destination.resolve():
        shutil.copy2(resolved[binary_name], destination)

giza_root = tools / 'giza-pp'
shutil.copy2(giza_root / 'mkcls-v2' / 'mkcls', external_bin / 'mkcls')
shutil.copy2(giza_root / 'GIZA++-v2' / 'GIZA++', external_bin / 'GIZA++')
shutil.copy2(giza_root / 'GIZA++-v2' / 'snt2cooc.out', external_bin / 'snt2cooc.out')

fast_align_root = tools / 'fast_align' / 'build'
shutil.copy2(fast_align_root / 'fast_align', external_bin / 'fast_align')
shutil.copy2(fast_align_root / 'atools', external_bin / 'atools')

print('Moses binaries staged in:', repo_bin)
for name, path in resolved.items():
    print(f'{name}: {path}')
print('External binaries staged in:', external_bin)


## 3. Download corpora

Selected corpora:

1. News Commentary v9
2. Europarl v7 English-German


In [ ]:
%%bash
set -e
RAW=/content/pbsmt_demo/data/raw
cd "$RAW"
[ -f training-parallel-nc-v9.tgz ] || wget -q https://www.statmt.org/wmt14/training-parallel-nc-v9.tgz
[ -f de-en.tgz ] || wget -q https://www.statmt.org/europarl/v7/de-en.tgz
mkdir -p nc europarl
tar -xzf training-parallel-nc-v9.tgz -C nc
tar -xzf de-en.tgz -C europarl


## 4. Create the 40k subset

This keeps the workflow small enough for a demo while still using real parallel data for both directions.


In [ ]:
import re
from pathlib import Path

raw = Path('/content/pbsmt_demo/data/raw')
proc = Path('/content/pbsmt_demo/data/processed')
proc.mkdir(parents=True, exist_ok=True)

news_en = raw / 'nc' / 'training' / 'news-commentary-v9.de-en.en'
news_de = raw / 'nc' / 'training' / 'news-commentary-v9.de-en.de'
euro_en = raw / 'europarl' / 'europarl-v7.de-en.en'
euro_de = raw / 'europarl' / 'europarl-v7.de-en.de'

assert news_en.exists(), news_en
assert news_de.exists(), news_de
assert euro_en.exists(), euro_en
assert euro_de.exists(), euro_de

def clean_line(text: str) -> str:
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def good_pair(src: str, tgt: str, max_len: int) -> bool:
    if not src or not tgt:
        return False
    if src.startswith('<') or tgt.startswith('<'):
        return False
    src_len = len(src.split())
    tgt_len = len(tgt.split())
    if src_len == 0 or tgt_len == 0:
        return False
    if src_len > max_len or tgt_len > max_len:
        return False
    ratio = max(src_len / tgt_len, tgt_len / src_len)
    if ratio > 2.5:
        return False
    return True

def collect_pairs(src_path: Path, tgt_path: Path, limit: int):
    pairs = []
    seen = set()
    with src_path.open('r', encoding='utf-8', errors='ignore') as src_f, tgt_path.open('r', encoding='utf-8', errors='ignore') as tgt_f:
        for src, tgt in zip(src_f, tgt_f):
            src = clean_line(src)
            tgt = clean_line(tgt)
            if not good_pair(src, tgt, CONFIG['max_sentence_length']):
                continue
            key = (src, tgt)
            if key in seen:
                continue
            seen.add(key)
            pairs.append(key)
            if len(pairs) >= limit:
                break
    if len(pairs) < limit:
        raise ValueError(f'Only found {len(pairs)} pairs in {src_path.name}, expected {limit}')
    return pairs

news_pairs = collect_pairs(news_en, news_de, CONFIG['news_pairs'])
euro_pairs = collect_pairs(euro_en, euro_de, CONFIG['europarl_pairs'])
all_pairs = news_pairs + euro_pairs
assert len(all_pairs) == CONFIG['total_pairs']

train_pairs = all_pairs[:CONFIG['train_pairs']]
valid_pairs = all_pairs[CONFIG['train_pairs']:CONFIG['train_pairs'] + CONFIG['valid_pairs']]
test_pairs = all_pairs[CONFIG['train_pairs'] + CONFIG['valid_pairs']:]

def write_split(name, pairs):
    with (proc / f'{name}.en').open('w', encoding='utf-8') as enf, (proc / f'{name}.de').open('w', encoding='utf-8') as def_:
        for en, de in pairs:
            enf.write(en + '\n')
            def_.write(de + '\n')

write_split('train', train_pairs)
write_split('valid', valid_pairs)
write_split('test', test_pairs)

print({
    'train': len(train_pairs),
    'valid': len(valid_pairs),
    'test': len(test_pairs),
})


## 5. Normalize, tokenize, truecase, and clean


In [ ]:
%%bash
set -e
BASE=/content/pbsmt_demo
MOSES=$BASE/tools/mosesdecoder
PROC=$BASE/data/processed
WORK=$BASE/data/work
mkdir -p "$WORK/tok" "$WORK/truecase"
for split in train valid test; do
  perl "$MOSES/scripts/tokenizer/normalize-punctuation.perl" -l en < "$PROC/$split.en" | perl "$MOSES/scripts/tokenizer/tokenizer.perl" -threads 2 -l en > "$WORK/tok/$split.tok.en"
  perl "$MOSES/scripts/tokenizer/normalize-punctuation.perl" -l de < "$PROC/$split.de" | perl "$MOSES/scripts/tokenizer/tokenizer.perl" -threads 2 -l de > "$WORK/tok/$split.tok.de"
done
perl "$MOSES/scripts/recaser/train-truecaser.perl" --model "$WORK/truecase/truecase-model.en" --corpus "$WORK/tok/train.tok.en"
perl "$MOSES/scripts/recaser/train-truecaser.perl" --model "$WORK/truecase/truecase-model.de" --corpus "$WORK/tok/train.tok.de"
for split in train valid test; do
  perl "$MOSES/scripts/recaser/truecase.perl" --model "$WORK/truecase/truecase-model.en" < "$WORK/tok/$split.tok.en" > "$WORK/$split.tc.en"
  perl "$MOSES/scripts/recaser/truecase.perl" --model "$WORK/truecase/truecase-model.de" < "$WORK/tok/$split.tok.de" > "$WORK/$split.tc.de"
done
perl "$MOSES/scripts/training/clean-corpus-n.perl" "$WORK/train.tc" en de "$WORK/train.clean" 1 50


## 6. Train language models

A German LM is used for `en->de`, and an English LM is used for `de->en`.


In [ ]:
%%bash
set -e
BASE=/content/pbsmt_demo
KENLM=$BASE/tools/kenlm/build/bin
WORK=$BASE/data/work
MODELS=$BASE/models
mkdir -p "$MODELS/lm"
"$KENLM/lmplz" -o 5 -S 50% --discount_fallback < "$WORK/train.clean.de" > "$MODELS/lm/de.arpa"
"$KENLM/build_binary" trie "$MODELS/lm/de.arpa" "$MODELS/lm/de.binary"
"$KENLM/lmplz" -o 5 -S 50% --discount_fallback < "$WORK/train.clean.en" > "$MODELS/lm/en.arpa"
"$KENLM/build_binary" trie "$MODELS/lm/en.arpa" "$MODELS/lm/en.binary"


## 7. Build fast_align alignments for both directions

The same cleaned parallel corpus is reused in both directions, but each direction gets its own alignment file with the source/target order matching the training direction.


In [ ]:
%%bash
set -e
BASE=/content/pbsmt_demo
MOSES=$BASE/tools/mosesdecoder
FAST_ALIGN=$BASE/tools/bin/fast_align
WORK=$BASE/data/work
MODELS=$BASE/models
ALIGN=grow-diag-final-and
perl "$MOSES/scripts/ems/support/prepare-fast-align.perl" "$WORK/train.clean.en" "$WORK/train.clean.de" > "$WORK/train.en-de.fa"
"$FAST_ALIGN" -i "$WORK/train.en-de.fa" -d -o -v > "$WORK/train.en-de.forward"
"$FAST_ALIGN" -r -i "$WORK/train.en-de.fa" -d -o -v > "$WORK/train.en-de.reverse"
perl "$MOSES/scripts/ems/support/symmetrize-fast-align.perl" "$WORK/train.en-de.forward" "$WORK/train.en-de.reverse" "$WORK/train.clean.en" "$WORK/train.clean.de" "$MODELS/aligned_en_de" "$ALIGN" "$MOSES/bin/symal"
perl "$MOSES/scripts/ems/support/prepare-fast-align.perl" "$WORK/train.clean.de" "$WORK/train.clean.en" > "$WORK/train.de-en.fa"
"$FAST_ALIGN" -i "$WORK/train.de-en.fa" -d -o -v > "$WORK/train.de-en.forward"
"$FAST_ALIGN" -r -i "$WORK/train.de-en.fa" -d -o -v > "$WORK/train.de-en.reverse"
perl "$MOSES/scripts/ems/support/symmetrize-fast-align.perl" "$WORK/train.de-en.forward" "$WORK/train.de-en.reverse" "$WORK/train.clean.de" "$WORK/train.clean.en" "$MODELS/aligned_de_en" "$ALIGN" "$MOSES/bin/symal"
ls -lh "$MODELS" | grep aligned || true


## 8. Train phrase-based models

To keep this demo smaller and faster, the notebook uses the precomputed `fast_align` word alignments and runs Moses training from steps `4-9`.


In [ ]:
%%bash
set -e
BASE=/content/pbsmt_demo
MOSES=$BASE/tools/mosesdecoder
WORK=$BASE/data/work
MODELS=$BASE/models
EXTBIN=$BASE/tools/bin
ALIGN=grow-diag-final-and
mkdir -p "$MODELS/en-de/model" "$MODELS/de-en/model"
perl "$MOSES/scripts/training/train-model.perl" \
  -root-dir "$MODELS/en-de" \
  -corpus "$WORK/train.clean" \
  -f en -e de \
  -alignment "$ALIGN" \
  -alignment-file "$MODELS/aligned_en_de" \
  -reordering msd-bidirectional-fe \
  -max-phrase-length 5 \
  -lm 0:5:"$MODELS/lm/de.binary":8 \
  -external-bin-dir "$EXTBIN" \
  -cores 2 \
  -parallel \
  -first-step 4 -last-step 9
perl "$MOSES/scripts/training/train-model.perl" \
  -root-dir "$MODELS/de-en" \
  -corpus "$WORK/train.clean" \
  -f de -e en \
  -alignment "$ALIGN" \
  -alignment-file "$MODELS/aligned_de_en" \
  -reordering msd-bidirectional-fe \
  -max-phrase-length 5 \
  -lm 0:5:"$MODELS/lm/en.binary":8 \
  -external-bin-dir "$EXTBIN" \
  -cores 2 \
  -parallel \
  -first-step 4 -last-step 9


## 9. Make model directories self-contained

This copies the language model binaries into each directional model directory and rewrites `moses.ini` to use the local file path.


In [ ]:
from pathlib import Path
import shutil

base = Path('/content/pbsmt_demo')
models = base / 'models'
mapping = {
    'en-de': models / 'lm' / 'de.binary',
    'de-en': models / 'lm' / 'en.binary',
}

for direction, lm_path in mapping.items():
    model_dir = models / direction / 'model'
    local_lm = model_dir / 'lm.binary'
    shutil.copy2(lm_path, local_lm)
    ini_path = model_dir / 'moses.ini'
    text = ini_path.read_text(encoding='utf-8')
    text = text.replace(str(model_dir) + '/', '')
    text = text.replace(str(lm_path), 'lm.binary')
    ini_path.write_text(text, encoding='utf-8')
    print(f'Updated {ini_path}')


## 10. Decode held-out test data and compute BLEU

This is a quick sanity check that both models load and translate.


In [ ]:
%%bash
set -e
BASE=/content/pbsmt_demo
MOSES=$BASE/tools/mosesdecoder/bin/moses
WORK=$BASE/data/work
MODELS=$BASE/models
EXPORTS=$BASE/exports
mkdir -p "$EXPORTS/test_outputs"
(cd "$MODELS/en-de/model" && "$MOSES" -f moses.ini < "$WORK/test.tc.en" > "$EXPORTS/test_outputs/pred.de")
(cd "$MODELS/de-en/model" && "$MOSES" -f moses.ini < "$WORK/test.tc.de" > "$EXPORTS/test_outputs/pred.en")
python3 -m sacrebleu "$WORK/test.tc.de" -i "$EXPORTS/test_outputs/pred.de" -m bleu -b -w 4 | tee "$EXPORTS/test_outputs/bleu_en_de.txt"
python3 -m sacrebleu "$WORK/test.tc.en" -i "$EXPORTS/test_outputs/pred.en" -m bleu -b -w 4 | tee "$EXPORTS/test_outputs/bleu_de_en.txt"


## 11. Package models

The archives below contain the self-contained directional model directories.


In [ ]:
%%bash
set -e
BASE=/content/pbsmt_demo
EXPORTS=$BASE/exports
cd "$BASE/models"
tar -czf "$EXPORTS/model_en_de.tar.gz" en-de/model
tar -czf "$EXPORTS/model_de_en.tar.gz" de-en/model
ls -lh "$EXPORTS"


## 12. Optional: download model archives


In [ ]:
from google.colab import files
files.download('/content/pbsmt_demo/exports/model_en_de.tar.gz')
files.download('/content/pbsmt_demo/exports/model_de_en.tar.gz')
